<a href="https://colab.research.google.com/github/aditya01ad/colab_codes/blob/main/notebooks/OR_Prep/Codes_ess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The ftory problem:
These are synthatic teaching inputs,vwith fractional production allowed and no demand caps.

| Product | Contribution margin | Machine 1 | Machine 2 |
| - | -------- | ------------ | ------------ |
| A | ₹40/unit | 2 hours/unit | 1 hour/unit  |
| B | ₹30/unit | 1 hour/unit  | 2 hours/unit |
| Available capacity | | 100 hours | 80 hours |\

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib as plt

from scipy.optimize import linprog

pd.set_option("display.precision", 4)

TOL = 1e-7

In [ ]:
# Storeinputs

products = ["A", "B"]

margin = {
    "A" : 40,
    "B" : 30
}
machine1_hours_per_unit = {
    "A" : 2,
    "B" : 1
}
machine2_hours_per_unit = {
    "A" : 1,
    "B" : 2
}
capacity = {
    "M1" : 100,
    "M2" : 80
}

In [ ]:
## Write functions to evaluate plan

# validate
def validate_plan(plan):
  # require one finite numeric quantity for each product.
  if set(plan) != set(products): #?
    raise ValueError(
        f"Plan must contain exactly these products: {products}"
    )

  for product in products:
    quantity = plan[product]

    if isinstance(quantity, (bool,np.bool_)):
      raise ValueError("Production quantities can not be Booleab.")

    if not isinstance(quantity, (int, float,np.floating)):
      raise ValueError(f"Quantity for {product} must be numerical.")
    if not np.isfinite(quantity):
      raise ValueError(f"Quantity for {product} must be finite.")

# Calculate total contribution margin in rupees
def objective(plan):
  validate_plan(plan)

  return sum(margin[product] * plan[product] for product in products)

# calculate machine hours requiredby plan
def used_resources(plan):
  validate_plan(plan)

  return {
    "M1" : sum(machine1_hours_per_unit[p] * plan[p] for p in products),
    "M2" : sum(machine2_hours_per_unit[p] * plan[p] for p in products)
  }

# check nonnegetivity and both capacity conatraints
def is_feasible(plan, tol=TOL):
  validate_plan(plan)

  if any(plan[p] < -tol for p in products):
    return False

  usage = used_resources(plan)

  return all(usage[resource] <= capacity[resource] + tol for resource in capacity)

In [ ]:
candidate = {
  "A":40,
  "B":20
}

print("Production plan:", candidate)
print("Contribution margin: rs. ", objective(candidate))
print("Resource usage:", used_resources(candidate))
print("Feasible:", is_feasible(candidate))

assert is_feasible(candidate)
assert objective(candidate) == 2200

Production plan: {'A': 40, 'B': 20}
Contribution margin: rs.  2200
Resource usage: {'M1': 100, 'M2': 80}
Feasible: True


## Pandas andmatrix construction
> Focus: _Represent inputs clearly and preserve product ordering._

In [ ]:
# Build the input table
items = pd.DataFrame({
  "product": products,
  "margin_inr": [margin[p] for p in products],
  "M1_hr_per_unit": [machine1_hours_per_unit[p] for p in products],
  "M2_hr_per_unit": [machine2_hours_per_unit[p] for p in products],
}).set_index("product")

print(items)

         margin_inr  M1_hr_per_unit  M2_hr_per_unit
product                                            
A                40               2               1
B                30               1               2


In [ ]:
# Validate the inputs

if not items.index.is_unique:
  raise ValueError("Duplicate product names.")
if items.isna().any().any():
  raise ValueErro("Missing model coefficients.")

numeric_inputs = items.to_numpy(dtype=float)

if not np.isfinite(numeric_inputs).all():
  raise ValueError("All coefficients must be finite.")

# This check reflects this factory example's assumptions.
if (numeric_inputs < 0).any():
  raise ValueError("This example expects nonnegativity input coefficients.")

capacity_values = np.array([capacity["M1"], capacity["M2"]], dtype=float)

if not np.isfinite(capacity_values).all():
  raise ValueError("Capacities must be finite.")
if (capacity_values < 0).any():
  raise ValueError("Capacities cannot be negetive.")

print("Input check passed.")

Input check passed.


In [ ]:
## Construct the matrix model

# Original MAXIMIZATION coefficients.
c_max = items["margin_inr"].to_numpy(dtype=float)

# Rows = machines, clmns = products
A = items[["M1_hr_per_unit", "M2_hr_per_unit"]].to_numpy(dtype=float)

# capacity vector in the same machine order as A's rows
b = capacity_values.copy()

# One nonnegative continuous variable per produuct
bounds = [(0, None)] * len(c_max)

print("Product order:", items.index.tolist())
print("\nc:")
print(c_max)

print("\nA:")
print(A)

print("\nb:")
print(b)

print("\nShapes:")
print("c:",c_max.shape)
print("A:", A.shape)
print("b:", b.shape)

assert A.shape == (2, 2)
assert c_max.shape == (2,)
assert b.shape ==(2,)

Product order: ['A', 'B']

c:
[40. 30.]

A:
[[2. 1.]
 [1. 2.]]

b:
[100.  80.]

Shapes:
c: (2,)
A: (2, 2)
b: (2,)


## Solve LP with SciPy:

In [ ]:
## call the solver — Doseminimization problem

# Maximization
result = linprog(
   c = -c_max,
   A_ub = A,
   b_ub = b,
   bounds = bounds,
   method = "highs"
)

print("Status code:", result.status)
print("Message:", result.message)

if not result.success:
    raise RuntimeError(f"Optimization faild:", result.message)

Status code: 0
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [ ]:
# Extract and interpret the solution

solution = pd.Series(
    result.x, index = items.index, name = "units"
)

optimal_margin =float(c_max@result.x)

usage = A@result.x
slack = b-usage

resource_report = pd.DataFrame({
    "capacity_hours":b,
    "used_hours":usage,
    "slack_hours":slack,
},index=["M1", "M2"])

print("Optimal quantities:")
print(solution)

print(
    f"\nOptimal contribution margin: ₹{optimal_margin:,.2f}"
)

print("\nResource report:")
print(resource_report)

Optimal quantities:
product
A    40.0
B    20.0
Name: units, dtype: float64

Optimal contribution margin: ₹2,200.00

Resource report:
    capacity_hours  used_hours  slack_hours
M1           100.0       100.0          0.0
M2            80.0        80.0          0.0


In [ ]:
## Check the numerical result

# Nonnegative production
assert np.all(result.x >= -TOL)

# Capacity feasibility
assert np.all(A@result.x <= b+TOL)

# Reconstructed objective agrees with solver objective
assert np.isclose(
    optimal_margin,
    -result.fun,
    atol=TOL
)

# Expected solution for this known teaching instance
assert np.allclose(
    result.x, [40, 20], atol=TOL
)

assert np.isclose(
    optimal_margin, 2200, atol=TOL
)

print("Baseline solution check passed.")

Baseline solution check passed.


In [ ]:
## Compare all vertices

vertices = np.array([
    [0, 0],
    [0, 40],
    [40, 20],
    [50, 0]
], dtype=float)

vertex_values = vertices @ c_max

vertex_usage = vertices @ A.T

vertex_feasibility = (
    np.all(vertices >= -TOL, axis=1) & np.all(vertex_usage <= b+TOL, axis=1)
)

vertex_report = pd.DataFrame({
    "A_units": vertices[:, 0],
    "B_units": vertices[:,1],
    "M1_used": vertex_usage[:, 0],
    "M2_used": vertex_usage[:, 1],
    "margin_inr": vertex_values,
    "feasible":vertex_feasibility
})

print(vertex_report.to_string(index=False))

assert vertex_feasibility.all()
assert np.isclose(vertex_values.max(), optimal_margin, atol=TOL)


 A_units  B_units  M1_used  M2_used  margin_inr  feasible
     0.0      0.0      0.0      0.0         0.0      True
     0.0     40.0     40.0     80.0      1200.0      True
    40.0     20.0    100.0     80.0      2200.0      True
    50.0      0.0    100.0     50.0      2000.0      True


In [ ]:
## Solve binding equations

intersection = np.linalg.solve(A, b)

print("Intersection:",intersection)
print("Margin:", c_max @ intersection)

assert np.allclose(intersection, result.x, atol=TOL)

Intersection: [40. 20.]
Margin: 2200.0


## Plot the feasinle region:

In [ ]:
# @title
## Prepare the boundry lines

x_grid = np.linspace(0, 55, 500)

# 2x + y = 100
y_machine1 = 100 - 2 * x_grid

# x + 2y = 80
y_machine2 = (80 - x_grid) / 2

y_upper = np.minimum(y_machine1, y_machine2)

feasible_x = (
    (y_machine1 >= 0)
    & (y_machine2 >= 0)
)

# Objective line through the optimal solution.
y_objective = (
    optimal_margin - c_max[0] * x_grid
) / c_max[1]

In [ ]:
## Draw the plot

